In [1]:
# Assuming you have already installed prfpy_csenf

%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
warnings.filterwarnings(action='ignore')
import os
opj = os.path.join
import time
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
#
from prfpy_csenf.model import CSenFModel
from prfpy_csenf.stimulus import CSenFStimulus
from prfpy_csenf.fit import CSenFFitter
from prfpy_csenf.rf import * 
from prfpy_csenf.csenf_plot_functions import *


In [49]:
# Settings
main_dir = '/Users/uriel/disks/meso_shared'
project_dir = 'nCSF'
subject = 'sub-01'
nCSF_ses = 'ses-01'
nCSF_task_name = 'nCSF'
TR = 1.6

In [40]:
# Load events files 
events_dir = '{}/{}/{}/{}/func'.format(main_dir, project_dir, subject, nCSF_ses)
events_fn = '{}/{}_{}_task-{}_dir-PA_run-01_events.tsv'.format(events_dir, subject, nCSF_ses, nCSF_task_name)
events_df = pd.read_table(events_fn, sep="\t")



In [25]:
sf_minFreq = 0.05
sf_maxFreq = 16
sf_filtNum = 6 

minCont = 0.25
maxCont = 80
contNum = 12

sf_filtCenters = np.concatenate([np.round(np.logspace(np.log10(0.05), np.log10(16), sf_filtNum), 2), [0]])

contValues = np.concatenate([np.logspace(np.log10(minCont), np.log10(maxCont), contNum), [0]])

In [44]:
mapping_SF = {i+1: sf_filtCenters[i] for i in range(len(sf_filtCenters))}
mapping_MC = {i+1: contValues[i] for i in range(len(contValues))}

events_df['spatial_frequency'] = events_df['spatial_frequency'].replace(mapping_SF)
events_df['michelson_contrast'] = events_df['michelson_contrast'].replace(mapping_MC)

In [45]:
events_df['spatial_frequency']

0      0.0
1      0.0
2      0.0
3      0.0
4      0.0
      ... 
209    0.0
210    0.0
211    0.0
212    0.0
213    0.0
Name: spatial_frequency, Length: 214, dtype: float64

In [46]:
sfs_seq = np.array(events_df['spatial_frequency'])
con_seq = np.array(events_df['michelson_contrast'])

In [47]:
sfs_seq

array([ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
        0.  ,  1.59,  1.59,  1.59,  1.59,  1.59,  1.59,  1.59,  1.59,
        1.59,  1.59,  1.59,  1.59,  1.59,  1.59,  1.59,  1.59,  1.59,
        1.59,  1.59,  1.59,  1.59,  1.59,  1.59,  1.59,  0.  ,  0.  ,
        0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.05,
        0.05,  0.05,  0.05,  0.05,  0.05,  0.05,  0.05,  0.05,  0.05,
        0.05,  0.05, 16.  , 16.  , 16.  , 16.  , 16.  , 16.  , 16.  ,
       16.  , 16.  , 16.  , 16.  , 16.  ,  0.  ,  0.  ,  0.  ,  0.  ,
        0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.16,  0.16,  0.16,
        0.16,  0.16,  0.16,  0.16,  0.16,  0.16,  0.16,  0.16,  0.16,
        0.16,  0.16,  0.16,  0.16,  0.16,  0.16,  0.16,  0.16,  0.16,
        0.16,  0.16,  0.16,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,  0.  ,
        0.  ,  0.  ,  0.  ,  0.  ,  5.05,  5.05,  5.05,  5.05,  5.05,
        5.05,  5.05,  5.05,  5.05,  5.05,  5.05,  5.05,  0.05,  0.05,
        0.05,  0.05,

In [50]:
# Create stim object
csenf_stim = CSenFStimulus(
    SF_seq  = sfs_seq, # np array, 1D 
    CON_seq = con_seq, # np array, 1D 
    TR      = TR,
    discrete_levels=True, # If discrete levels of SF and contrast. Default is true
    )

Number of timepoints: 214
Number of unique SF levels: 6, [ 0.05  0.16  0.5   1.59  5.05 16.  ]
Number of unique CON levels: 12, [ 0.25   0.422  0.714  1.205  2.037  3.441  5.813  9.82  16.591 28.029
 47.353 80.   ]


In [54]:
csenf_model = CSenFModel(
    stimulus = csenf_stim,    
    hrf=[1,1,0],
    edge_type='CRF',
)

TypeError: spm_hrf() got an unexpected keyword argument 'tr'. Did you mean 't_r'?